In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/bank_churn_clean.csv")
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
df_encoded = pd.get_dummies(df, columns=["Geography"], drop_first=True, dtype=int)
df_encoded.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain
0,619,Female,42,2,0.00,1,1,1,101348.88,1,0,0
1,608,Female,41,1,83807.86,1,0,1,112542.58,0,0,1
2,502,Female,42,8,159660.80,3,1,0,113931.57,1,0,0
3,699,Female,39,1,0.00,2,0,0,93826.63,0,0,0
4,850,Female,43,2,125510.82,1,1,1,79084.10,0,0,1


In [4]:
df_encoded = pd.get_dummies(df_encoded, columns=["Gender"], drop_first=True, dtype=int)
df_encoded.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0


In [6]:
print(df_encoded.shape)
print(list(df_encoded.columns))

(10000, 12)
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_Germany', 'Geography_Spain', 'Gender_Male']


In [7]:
df_encoded["BalanceSalaryRatio"] = df_encoded["Balance"] / df_encoded["EstimatedSalary"]
df_encoded["BalanceSalaryRatio"].describe().round(2)

count    10000.00
mean         3.88
std        108.34
min          0.00
25%          0.00
50%          0.75
75%          1.51
max      10614.66
Name: BalanceSalaryRatio, dtype: float64

In [8]:
extreme = df_encoded[df_encoded["BalanceSalaryRatio"] > 10]
print(len(extreme))
extreme[["Balance", "EstimatedSalary", "BalanceSalaryRatio", "Exited"]].sort_values("BalanceSalaryRatio", ascending=False).head(10)

357


,Balance,EstimatedSalary,BalanceSalaryRatio,Exited
2362,122917.71,11.58,10614.655440,1
9010,121669.93,91.75,1326.102779,1
7565,105355.81,123.07,856.064110,1
8722,87619.29,143.34,611.268941,0
1462,103560.57,236.45,437.980842,1
3446,87374.88,247.36,353.229625,0
2046,120018.86,343.38,349.521987,0
6956,180614.04,523.00,345.342333,0
5120,141462.13,440.20,321.358769,0
195,174790.15,600.36,291.142231,0


In [9]:
print((df_encoded["BalanceSalaryRatio"] > 10).sum())
df_encoded["BalanceSalaryRatio"].quantile([0.5, 0.9, 0.95, 0.99, 0.999]).round(2)

357


0.500      0.75
0.900      3.67
0.950      7.25
0.990     35.48
0.999    276.80
Name: BalanceSalaryRatio, dtype: float64

In [10]:
import numpy as np

df_encoded["BalanceSalaryRatio_log"] = np.log1p(df_encoded["BalanceSalaryRatio"])
df_encoded["BalanceSalaryRatio_log"].describe().round(2)

count    10000.00
mean         0.66
std          0.77
min          0.00
25%          0.00
50%          0.56
75%          0.92
max          9.27
Name: BalanceSalaryRatio_log, dtype: float64

In [11]:
df_encoded = df_encoded.drop(columns=["BalanceSalaryRatio"])
print(df_encoded.shape)
print(list(df_encoded.columns))

(10000, 13)
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_Germany', 'Geography_Spain', 'Gender_Male', 'BalanceSalaryRatio_log']


In [12]:
df_encoded["ProductDensity"] = df_encoded["NumOfProducts"] / (df_encoded["Tenure"] + 1)
df_encoded["ProductDensity"].describe().round(2)

count    10000.00
mean         0.37
std          0.34
min          0.09
25%          0.17
50%          0.25
75%          0.50
max          3.00
Name: ProductDensity, dtype: float64

In [13]:
df_encoded["EngagementProduct"] = df_encoded["IsActiveMember"] * df_encoded["NumOfProducts"]
df_encoded["EngagementProduct"].value_counts().sort_index()

EngagementProduct
0    4849
1    2563
2    2446
3     113
4      29
Name: count, dtype: int64

In [14]:
df_encoded["TenureByAge"] = df_encoded["Tenure"] / df_encoded["Age"]
df_encoded["TenureByAge"].describe().round(3)

count    10000.000
mean         0.138
std          0.090
min          0.000
25%          0.065
50%          0.129
75%          0.200
max          0.556
Name: TenureByAge, dtype: float64

In [15]:
joined_age = df_encoded["Age"] - df_encoded["Tenure"]
print((joined_age < 18).sum())
df_encoded[joined_age < 18][["Age", "Tenure", "Exited"]].head(10)

322


,Age,Tenure,Exited
17,24,9,0
69,21,8,0
98,22,8,0
129,24,9,0
145,21,5,0
160,22,6,0
166,24,7,1
169,24,9,0
233,20,3,0
248,24,8,0


In [16]:
joined_age = df_encoded["Age"] - df_encoded["Tenure"]
print((joined_age < 18).sum())
print(joined_age.min())

322
8


In [17]:
df_encoded.to_csv("../data/processed/bank_churn_features.csv", index=False)

check = pd.read_csv("../data/processed/bank_churn_features.csv")
print(check.shape)

(10000, 16)
